In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rc
rc('animation', html='jshtml')

In [ ]:
# from skyfield.api import load_file, load
# import numpy as np

# # Load local ephemeris
# planets = load_file("de440s.bsp")

# sun = planets["sun"]
# earth = planets["earth"]
# moon = planets["moon"]

# ts = load.timescale()

In [ ]:
# One position per day
times = ts.utc(2025, 1, np.arange(1, 366))

earth_pos = earth.at(times).position.km
moon_pos = moon.at(times).position.km
sun_pos = sun.at(times).position.km

earth_vel = earth.at(times[0]).velocity.km_per_s
moon_vel = moon.at(times[0]).velocity.km_per_s
sun_vel = sun.at(times[0]).velocity.km_per_s

In [ ]:
def fibonacci_sphere(samples, amplitude, xc, yc, zc):

    points = []
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle in radians

    for i in range(samples):
        y = amplitude[i]*(1 - (i / float(samples - 1)) * 2)  # y goes from 1 to -1
        radius = np.sqrt(amplitude[i]**2 - y * y)  # radius at y

        theta = phi * i  # golden angle increment

        x = np.cos(theta) * radius
        z = np.sin(theta) * radius

        points.append((x+xc, y+yc, z+zc))

    return np.transpose(points)

In [ ]:
def fibonacci_sphere_filled(samples, amplitude, xc, yc, zc):
    points = []
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle in radians

    for i in range(samples):
        # 1. Distribute points evenly across standard surface height slices
        y_surface = (1 - (i / float(samples - 1)) * 2)
        radius_surface = np.sqrt(1.0 - y_surface**2)
        theta = phi * i

        # 2. Calculate the surface coordinate vector (unit sphere)
        x_unit = np.cos(theta) * radius_surface
        y_unit = y_surface
        z_unit = np.sin(theta) * radius_surface

        # 3. Pull the point inward by a variable fraction (cube root ensures uniform volume density)
        # Using a deterministic fraction based on 'i' so it remains a reproducible sequence
        fraction = (i / float(samples)) ** (1.0 / 3.0) 
        
        # 4. Scale by the maximum amplitude allowed for this particle index
        current_amp = amplitude[i] * fraction

        # 5. Project out to the final 3D volume coordinate
        x = x_unit * current_amp
        y = y_unit * current_amp
        z = z_unit * current_amp

        points.append((x + xc, y + yc, z + zc))

    return np.transpose(points)


In [ ]:
# 1. SETUP FIBONACCI SPHERES IN AU (1 AU = 149600000 km)
Scale_factor = 1000
KM_TO_AU = 1 / 149600000

mass_scale = 1E30

Nbody = 100
xyz = np.linspace(-1, 1, Nbody)

M = np.array([1.989E2,5.972E-2,7.346E-4])

# Constant amplitude shapes (radii converted to AU)
# Note: For strict visibility, you can multiply these A amplitudes by a scale factor like 10 or 50
A1 = M[0] * np.ones(Nbody)   # Sun Radius
A2 = M[1] * np.ones(Nbody)     # Earth Radius
A3 = M[2] * np.ones(Nbody)     # Moon Radius

# Generate positions using AU coordinates
s1 = fibonacci_sphere_filled(Nbody, A1, sun_pos[0][0] * KM_TO_AU * Scale_factor, 
                             sun_pos[1][0] * KM_TO_AU * Scale_factor, sun_pos[2][0] * KM_TO_AU * Scale_factor)                        # Sun at center
s2 = fibonacci_sphere_filled(Nbody, A2, earth_pos[0][0] * KM_TO_AU * Scale_factor, 
                             earth_pos[1][0] * KM_TO_AU * Scale_factor, earth_pos[2][0] * KM_TO_AU * Scale_factor)   # Earth at 1 AU
s3 = fibonacci_sphere_filled(Nbody, A3, moon_pos[0][0] * KM_TO_AU * Scale_factor, 
                             moon_pos[1][0] * KM_TO_AU * Scale_factor, moon_pos[2][0] * KM_TO_AU * Scale_factor)   # Earth at 1 AU


fig = plt.figure()
ax = plt.axes(projection='3d')

ax.scatter3D(s1[0],s1[1],s1[2])
ax.scatter3D(s2[0],s2[1],s2[2])
ax.scatter3D(s3[0],s3[1],s3[2])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from numba import njit, prange

fig = plt.figure()
ax = plt.axes(projection='3d')

bodies = [s1, s2, s3]
amps   = [A1, A2, A3]
colors = ['r', 'b', 'g', 'm']

Ns = [b.shape[1] for b in bodies]
P = np.vstack([b.T for b in bodies])
V = np.zeros_like(P)
AMP = np.concatenate(amps)
body_id = np.concatenate([np.full(Ns[i], i) for i in range(len(bodies))])

# v1 = np.zeros_like(s1).T  # Shape: (N1, 3)
# v1[:, 0] = 0            # Set x-velocity for body 1

# v2 = np.zeros_like(s2).T  # Shape: (N2, 3)
# v2[:, :] = [0,3,0]         # Set x-velocity for body 2

# v3 = np.zeros_like(s3).T  # Shape: (N3, 3)
# v3[:, :] = [-0.01,3,0]           # Set y-velocity for body 3

# Define initial velocity vectors for each body/particle
# Example: s1 moving right, s2 moving left, s3 stationary
v1 = np.zeros_like(s1).T  # Shape: (N1, 3)
v1[:, :] = sun_vel/10           # Set x-velocity for body 1

v2 = np.zeros_like(s2).T  # Shape: (N2, 3)
v2[:, :] = earth_vel/10         # Set x-velocity for body 2

v3 = np.zeros_like(s3).T  # Shape: (N3, 3)
v3[:, :] = moon_vel/10           # Set y-velocity for body 3

velocities = [v1, v2, v3]
# Replace V = np.zeros_like(P) with this:
V = np.vstack(velocities)

time_steps = 5
G = 1.994E-44 # 25
softening2 = 25.0**2 *0
damping = 0.995
dt = 1.0
VMAX = 1000.0

xlim, ylim, zlim = [-1000, 100], [-100, 1000], [-50, 500]
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_zlim(zlim)

masks = [body_id == i for i in range(len(bodies))] # set body equal to number to view individual object
scatters = [
    ax.scatter3D(P[m, 0], P[m, 1], P[m, 2], color=colors[i], s=4)
    for i, m in enumerate(masks)
]

@njit(parallel=True, fastmath=True, cache=True)
def compute_accel(P, AMP, G, softening2, Nbody, M, mass_scale):
    N = P.shape[0]
    accel = np.zeros((N, 3))
    absAMP = np.abs(AMP*mass_scale)
    for i in prange(N):
        ax_ = ay_ = az_ = 0.0
        for j in range(N):
            if i == j:
                continue
            dx = P[j, 0] - P[i, 0]
            dy = P[j, 1] - P[i, 1]
            dz = P[j, 2] - P[i, 2]
            dist2 = dx*dx + dy*dy + dz*dz + softening2
            dist = np.sqrt(dist2)
            f = G * absAMP[j] / (dist2 * dist)
            ax_ += f * dx; ay_ += f * dy; az_ += f * dz
        accel[i, 0] = ax_ / Nbody
        accel[i, 1] = ay_ / Nbody
        accel[i, 2] = az_ / Nbody
    return accel

_ = compute_accel(P, AMP, G, softening2, Nbody, M, mass_scale)  # warm up JIT before animation starts

def frame(w):
    global P, V

    for _ in range(time_steps):
        accel = compute_accel(P, AMP, G, softening2, Nbody, M, mass_scale)
        V[:] = (V + accel*dt) #* damping #np.clip((V + accel*dt) * damping, -VMAX, VMAX)
        P[:] += V*dt

    for sc, m in zip(scatters, masks):
        sc._offsets3d = (P[m,0], P[m,1], P[m,2])

    return scatters

plt.close()
anim = animation.FuncAnimation(fig, frame, frames=200, blit=False, repeat=True)

In [ ]:
#anim.save('quantum_gravity_simulation.gif', writer='pillow', fps=30, dpi=200)

In [ ]:
anim

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

ax.set_aspect('equal')
ax.grid(True)

# Calculate limits
all_x = np.concatenate([earth_pos[0], moon_pos[0], sun_pos[0]])
all_y = np.concatenate([earth_pos[1], moon_pos[1], sun_pos[1]])

padding = 0.05 * max(all_x.max()-all_x.min(), all_y.max()-all_y.min())

ax.set_xlim(all_x.min()-padding, all_x.max()+padding)
ax.set_ylim(all_y.min()-padding, all_y.max()+padding)

# Orbit trails
earth_path, = ax.plot([], [], 'b-', lw=1)
moon_path, = ax.plot([], [], '0.6', lw=1)
sun_path, = ax.plot([], [], 'y-', lw=1)

# Bodies
earth_dot, = ax.plot([], [], 'bo', markersize=8, label='Earth')
moon_dot, = ax.plot([], [], 'ko', markersize=4, label='Moon')
sun_dot, = ax.plot([], [], 'yo', markersize=12, label='Sun')

ax.legend()

def init():
    earth_path.set_data([], [])
    moon_path.set_data([], [])
    sun_path.set_data([], [])

    earth_dot.set_data([], [])
    moon_dot.set_data([], [])
    sun_dot.set_data([], [])

    return (
        earth_path,
        moon_path,
        sun_path,
        earth_dot,
        moon_dot,
        sun_dot,
    )

def update(frame):

    earth_path.set_data(earth_pos[0][:frame], earth_pos[1][:frame])
    moon_path.set_data(moon_pos[0][:frame], moon_pos[1][:frame])
    sun_path.set_data(sun_pos[0][:frame], sun_pos[1][:frame])

    earth_dot.set_data([earth_pos[0][frame]], [earth_pos[1][frame]])
    moon_dot.set_data([moon_pos[0][frame]], [moon_pos[1][frame]])
    sun_dot.set_data([sun_pos[0][frame]], [sun_pos[1][frame]])

    return (
        earth_path,
        moon_path,
        sun_path,
        earth_dot,
        moon_dot,
        sun_dot,
    )

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(earth_pos[0]),
    init_func=init,
    interval=25,
    blit=True
)

plt.close(fig)
ani